# Analisis Agregat: Seluruh Repetisi Pengujian

`testing_analysis.ipynb` membedah **satu** repetisi secara mendalam. Notebook ini
memuat **seluruh** repetisi yang ada di `data/testing/60-episodes/` sekaligus, agar
pertanyaan "apakah pola yang terlihat pada satu run memang berulang?" bisa dijawab.

| Batch | Tanggal | Run RL | Baseline HPA |
|---|---|---|---|
| A | 2026-08-07 | `1`, `2`, `3` | `hpa` (3 deployment paralel) |
| B | 2026-08-21 | `1.1`, `2.1`, `3.1` | `hpa-2` (3 deployment paralel) |

Total: 6 run per metode RL, dan 6 berkas HPA (2 folder x 3 deployment yang berjalan
bersamaan).

**Dua batch tidak setara.** Batch B jauh lebih cepat di median (~27-65 ms vs
~127-290 ms pada k6) dan di P90. Karena itu setiap ringkasan di bawah ditampilkan
per batch di samping nilai gabungannya: rata-rata gabungan berguna sebagai ringkasan,
tetapi bukan enam sampel dari satu populasi yang sama.

Angka k6 ditranskripsikan dari `image.png` tiap folder ke `k6_results.py`, satu
sumber yang dipakai bersama notebook ini dan `testing_analysis.ipynb`.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import importlib

import k6_results
# The notebook and k6_results.py are edited side by side; reload on every run so a
# change to the module does not need a kernel restart to show up here.
importlib.reload(k6_results)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

# This notebook covers the ORIGINAL campaign (batches A and B). The rerun lives in
# 'data/testing/60-episodes-2' (batch C) and reuses the run folder names '1', '2',
# '3' and 'hpa', so every k6_results lookup is keyed by dataset first.
DATA_DIR = 'data/testing/60-episodes'
DATASET = k6_results.dataset_of(DATA_DIR)
SLO_THRESHOLD = k6_results.SLO_MS   # 1000 ms, = MAX_RESPONSE_TIME in agent-dipa/.env

METHODS = k6_results.METHODS
LABELS = k6_results.METHOD_LABELS
# Same palette as testing_analysis.ipynb so figures from the two notebooks sit
# together in one document. Note: teal/pink and orange/pink are hard to separate
# under red-green colour blindness, so nothing below relies on colour alone -
# every series is also direct-labelled, marked, or given its own panel.
COLORS = {'ql': '#26A69A', 'qcrisp': '#EC407A', 'qfuzzy': '#5C6BC0', 'hpa': '#FF7043'}
MARKERS = {'ql': 'o', 'qcrisp': '^', 'qfuzzy': 's', 'hpa': 'D'}
BATCH_MARKERS = {'A': 'o', 'B': '^'}

# --- Enumerate every run -----------------------------------------------------
# An RL run folder holds one CSV per method. An HPA folder holds one CSV per
# HPA-managed deployment, all three under load at the same time: three
# deployments, not three repetitions, so they are kept as instances of one run.
def resolve(folder, method, instance=1):
    matches = sorted(glob.glob(f'{DATA_DIR}/{folder}/{k6_results.CSV_PATTERNS[method]}'))
    if len(matches) < instance:
        raise FileNotFoundError(
            f'No CSV #{instance} matching {k6_results.CSV_PATTERNS[method]} '
            f'in {DATA_DIR}/{folder}')
    return matches[instance - 1]


specs = []
for batch, folders in k6_results.RL_RUNS[DATASET].items():
    for folder in folders:
        for method in ('ql', 'qcrisp', 'qfuzzy'):
            specs.append({'method': method, 'folder': folder, 'batch': batch,
                          'instance': 1, 'run_id': folder,
                          'path': resolve(folder, method)})
    hpa_folder = k6_results.HPA_RUNS[DATASET][batch]
    for instance in (1, 2, 3):
        specs.append({'method': 'hpa', 'folder': hpa_folder, 'batch': batch,
                      'instance': instance, 'run_id': f'{hpa_folder}#{instance}',
                      'path': resolve(hpa_folder, 'hpa', instance)})

# --- Load, then trim every run to ONE common horizon --------------------------
# The files last 30.9-35.4 min, but they all stop carrying traffic at 30.53-30.75
# min: the 30-minute k6 profile plus the collector's lag. The difference is idle
# tail, and how much of it a file has depends only on when the collector was
# stopped by hand - see k6_results.last_traffic_elapsed_s. Those windows measure
# nothing yet still count toward replica-minutes and pull the utilisation means
# down, so the horizon ends at the earliest last-traffic window across all runs.
# That also fixes the coverage problem: the sampling cadence differs per run
# (~16 s/row for RL, ~17 s/row for HPA), so an equal row count would not mean
# equal wall-clock coverage of the load profile.
frames = {}
for spec in specs:
    df = pd.read_csv(spec['path'], parse_dates=['timestamp'])
    df['elapsed_s'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    frames[(spec['method'], spec['run_id'])] = df

last_traffic = {key: k6_results.last_traffic_elapsed_s(df) for key, df in frames.items()}
file_end = {key: df['elapsed_s'].max() for key, df in frames.items()}

HORIZON_S = min(last_traffic.values())
for key, df in frames.items():
    df = df.loc[df['elapsed_s'] <= HORIZON_S].reset_index(drop=True)
    df['elapsed_min'] = df['elapsed_s'] / 60.0
    frames[key] = df

RL_FOLDERS = [f for folders in k6_results.RL_RUNS[DATASET].values() for f in folders]
HPA_FOLDERS = list(k6_results.HPA_RUNS[DATASET].values())

print(f'Loaded {len(frames)} runs '
      f'({len(RL_FOLDERS)} RL run folders x 3 methods + '
      f'{len(HPA_FOLDERS)} HPA folders x 3 deployments)')
print(f'Files end at        : {min(file_end.values()) / 60:.2f}-'
      f'{max(file_end.values()) / 60:.2f} min')
print(f'Traffic ends at     : {min(last_traffic.values()) / 60:.2f}-'
      f'{max(last_traffic.values()) / 60:.2f} min')
print(f'Common horizon      : {HORIZON_S:.0f}s ({HORIZON_S / 60:.2f} min), '
      f'dropping up to {(max(file_end.values()) - HORIZON_S) / 60:.1f} min of idle tail')

## 1. Inventaris dan Statistik per Run

Satu baris per berkas CSV. Semua kolom dihitung pada horizon bersama di atas.

- **windows** - jendela observasi dengan trafik (`response_time_ms` bukan NaN dan > 0).
  Jendela tanpa trafik dibuang karena tidak mengukur apa pun.
- **rt_mean / rt_med** - rata-rata dan median dari P90 per jendela (sisi server).
- **slo_%** - persentase jendela yang P90-nya melampaui 1000 ms.
- **rep_min** (replica-minutes) - integral jumlah replica terhadap waktu, proksi biaya.

In [ ]:
def run_stats(method, run_id, folder, batch, instance, path):
    df = frames[(method, run_id)]
    rt = df['response_time_ms']
    valid = rt.notna() & (rt > 0)
    rt_valid = rt[valid]

    dt = df['elapsed_min'].diff().fillna(0.0)
    held = df['replica_state'].shift(1).fillna(df['replica_state'].iloc[0])
    diff = df['replica_state'].diff()

    return {
        'method': method, 'label': LABELS[method], 'run_id': run_id,
        'folder': folder, 'batch': batch, 'instance': instance,
        'file': path.split('/')[-1],
        'rows': len(df), 'dur_min': df['elapsed_min'].iloc[-1],
        'windows': int(valid.sum()),
        'rt_mean': rt_valid.mean(), 'rt_med': rt_valid.median(),
        'rt_max': rt_valid.max(),
        'slo_pct': (rt_valid > SLO_THRESHOLD).mean() * 100,
        'rep_mean': df['replica_state'].mean(),
        'rep_max': int(df['replica_state'].max()),
        'rep_min_count': int(df['replica_state'].min()),
        'ups': int((diff > 0).sum()), 'downs': int((diff < 0).sum()),
        'rep_minutes': float((held * dt).sum()),
        'cpu_mean': df['cpu_usage'].mean(), 'mem_mean': df['memory_usage'].mean(),
        'rps_mean': df['request_rate_rps'].mean(),
    }


per_run = pd.DataFrame([run_stats(**spec) for spec in specs])
per_run['method'] = pd.Categorical(per_run['method'], categories=METHODS, ordered=True)
per_run = per_run.sort_values(['batch', 'folder', 'method', 'instance']).reset_index(drop=True)

print(f'{"PER-RUN STATISTICS (common horizon " + f"{HORIZON_S / 60:.1f}" + " min)":^108}')
print('=' * 108)
print(f'{"Batch":<6} {"Run":<8} {"Method":<18} {"win":>5} {"rt_mean":>9} {"rt_med":>8} '
      f'{"slo_%":>7} {"rep_mean":>9} {"ups":>5} {"downs":>6} {"rep_min":>9}')
print('-' * 108)
current = None
for _, r in per_run.iterrows():
    if current is not None and r['batch'] != current:
        print('-' * 108)
    current = r['batch']
    print(f'{r["batch"]:<6} {r["run_id"]:<8} {r["label"]:<18} {r["windows"]:>5} '
          f'{r["rt_mean"]:>9.0f} {r["rt_med"]:>8.0f} {r["slo_pct"]:>7.1f} '
          f'{r["rep_mean"]:>9.2f} {r["ups"]:>5} {r["downs"]:>6} {r["rep_minutes"]:>9.1f}')
print('-' * 108)

per_run[['batch', 'run_id', 'label', 'file', 'rows', 'dur_min', 'windows']].round(1)

## 2. Hasil P90 k6 per Run

`http_req_duration` sisi klien, seluruh request dalam satu proses k6, dibandingkan
dengan threshold `p(90)<1000` yang diuji k6 sendiri.

Untuk folder HPA, tiga proses k6 berjalan bersamaan (satu per deployment), sehingga
batangnya adalah rata-rata ketiganya dan titik-titik menunjukkan nilai masing-masing.

In [ ]:
k6 = k6_results.k6_frame(DATASET)

fig, ax = plt.subplots(figsize=(15, 6))

folders = RL_FOLDERS + HPA_FOLDERS
xs = np.arange(len(folders))
width = 0.8 / 3

for i, method in enumerate(('ql', 'qcrisp', 'qfuzzy')):
    values, positions = [], []
    for x, folder in enumerate(folders):
        row = k6[(k6['method'] == method) & (k6['run'] == folder)]
        if len(row):
            values.append(row['p90'].iloc[0])
            positions.append(x + (i - 1) * width)
    ax.bar(positions, values, width, label=LABELS[method], color=COLORS[method], alpha=0.9)
    for x, v in zip(positions, values):
        ax.text(x, v + 40, f'{v:,.0f}', ha='center', va='bottom', fontsize=8)

# HPA folders: one bar for the mean of the three concurrent processes, plus a dot
# per process so the spread between deployments stays visible.
for x, folder in zip(xs[len(RL_FOLDERS):], HPA_FOLDERS):
    rows = k6[k6['run'] == folder]
    ax.bar([x], [rows['p90'].mean()], width * 1.6, label='HPA' if folder == HPA_FOLDERS[0] else None,
           color=COLORS['hpa'], alpha=0.9)
    ax.scatter([x] * len(rows), rows['p90'], color='white', edgecolor='#37474F',
               zorder=5, s=28, linewidth=1.2,
               label='per deployment' if folder == HPA_FOLDERS[0] else None)
    ax.text(x, rows['p90'].mean() + 40, f'{rows["p90"].mean():,.0f}',
            ha='center', va='bottom', fontsize=8)

ax.axhline(SLO_THRESHOLD, color='red', linestyle='--', linewidth=2, alpha=0.8,
           label=f'k6 threshold p(90) < {SLO_THRESHOLD:.0f} ms')
ax.axvline(len(k6_results.RL_RUNS[DATASET]['A']) - 0.5, color='#455A64', linewidth=1)
ax.axvline(len(RL_FOLDERS) - 0.5, color='#455A64', linewidth=1)

ax.set_ylim(0, k6['p90'].max() * 1.18)
for x_mid, batch in ((1, 'A'), (4, 'B')):
    ax.text(x_mid, ax.get_ylim()[1] * 0.95,
            f'batch {batch}  ({k6_results.BATCH_DATES[batch]})',
            ha='center', fontsize=9, style='italic')

ax.set_xticks(xs)
ax.set_xticklabels(folders)
ax.set_xlabel('Run folder')
ax.set_ylabel('k6 http_req_duration P90 (ms)')
ax.set_title('k6 P90 per Run', fontsize=14, fontweight='bold')
# Below the axes: the legend would otherwise sit on top of the tallest bars.
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncols=6, fontsize=9)

plt.tight_layout()
plt.show()

pivot = (k6.pivot_table(index='run', columns='label', values='p90', aggfunc='mean')
           .reindex(folders)[[LABELS[m] for m in METHODS if LABELS[m] in
                              k6['label'].unique()]])
print('k6 P90 per run (ms); HPA column is the mean of its three concurrent processes')
pivot.round(0)

## 3. Rata-rata P90 k6 per Metode

Nilai yang diminta: satu angka per metode, rata-rata P90 k6 atas seluruh run-nya.
Titik-titik menunjukkan tiap run individual, karena rata-rata dari enam run yang
tersebar 897-1810 ms menyembunyikan lebih banyak daripada yang ditampilkannya.

In [ ]:
overall = (k6.groupby('method', observed=True)['p90']
             .agg(mean='mean', sd='std', n='count', lo='min', hi='max'))
overall = overall.reindex(METHODS)

by_batch = k6.pivot_table(index='method', columns='batch', values='p90',
                          aggfunc='mean').reindex(METHODS)

fig, ax = plt.subplots(figsize=(13, 6))
xs = np.arange(len(METHODS))

bars = ax.bar(xs, overall['mean'], width=0.55,
              color=[COLORS[m] for m in METHODS], alpha=0.85)
ax.errorbar(xs, overall['mean'], yerr=overall['sd'].fillna(0), fmt='none',
            ecolor='#37474F', capsize=6, linewidth=1.4)

for bar, method, value in zip(bars, METHODS, overall['mean']):
    if value >= SLO_THRESHOLD:
        bar.set_hatch('//')
        bar.set_edgecolor('white')

# Every individual run behind the average, batch A as circles and batch B as triangles.
for x, method in zip(xs, METHODS):
    rows = k6[k6['method'] == method]
    for batch, marker in BATCH_MARKERS.items():
        sub = rows[rows['batch'] == batch]
        jitter = np.linspace(-0.16, 0.16, len(sub)) if len(sub) > 1 else [0.0]
        ax.scatter(x + np.array(jitter), sub['p90'], marker=marker, s=46,
                   color='white', edgecolor='#263238', linewidth=1.2, zorder=5,
                   label=f'run in batch {batch}' if x == 0 else None)

ax.axhline(SLO_THRESHOLD, color='red', linestyle='--', linewidth=2, alpha=0.8,
           label=f'k6 threshold p(90) < {SLO_THRESHOLD:.0f} ms')

for x, method, value in zip(xs, METHODS, overall['mean']):
    verdict = 'PASS' if value < SLO_THRESHOLD else 'FAIL'
    ax.text(x, value + overall['sd'].fillna(0).iloc[list(METHODS).index(method)] + 70,
            f'{value:,.0f} ms\n{verdict}', ha='center', va='bottom',
            fontsize=11, fontweight='bold',
            color='#2E7D32' if verdict == 'PASS' else '#C62828')

ax.set_xticks(xs)
ax.set_xticklabels([LABELS[m] for m in METHODS])
ax.set_ylabel('k6 http_req_duration P90 (ms)')
ax.set_title('Rata-rata P90 k6 per Metode (seluruh run, +/- 1 sd)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, (overall['mean'] + overall['sd'].fillna(0)).max() * 1.35)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f'{"MEAN k6 P90 PER METHOD":^92}')
print('=' * 92)
print(f'{"Method":<20} {"runs":>5} {"mean":>10} {"sd":>9} {"min":>9} {"max":>9} '
      f'{"batch A":>10} {"batch B":>10}')
print('-' * 92)
for method in METHODS:
    r = overall.loc[method]
    print(f'{LABELS[method]:<20} {int(r["n"]):>5} {r["mean"]:>7.0f} ms {r["sd"]:>6.0f} ms '
          f'{r["lo"]:>6.0f} ms {r["hi"]:>6.0f} ms '
          f'{by_batch.loc[method, "A"]:>7.0f} ms {by_batch.loc[method, "B"]:>7.0f} ms')
print('-' * 92)
passing = k6[k6['passes_slo']]
names = ', '.join(
    f'{r["label"]} run {r["run"]}' + (f'#{r["instance"]}' if r['method'] == 'hpa' else '')
    for _, r in passing.iterrows())
print(f'k6 processes meeting p(90)<{SLO_THRESHOLD:.0f}ms: {len(passing)} of {len(k6)}'
      + (f'  ({names})' if len(passing) else ''))

overall.round(1)

## 4. Response Time Sisi Server di Seluruh Run

Kolom `response_time_ms` adalah rata-rata P90 per endpoint untuk satu jendela
observasi (~15 detik) - inilah angka yang dilihat agent sebagai state dan reward,
berbeda dari P90 k6 di atas.

Pola pembacaan yang dipakai pada seluruh sisa notebook: batang adalah rata-rata
antar run, garis galat adalah +/- 1 simpangan baku antar run, dan setiap run
ditampilkan sebagai titik (lingkaran = batch A, segitiga = batch B).

In [ ]:
def aggregate(column):
    """Mean, sd and n of `column` across the runs of each method."""
    return (per_run.groupby('method', observed=True)[column]
            .agg(mean='mean', sd='std', n='count')
            .reindex(METHODS))


def bar_with_runs(ax, column, ylabel, title, fmt='{:,.0f}', ref=None, ref_label=None):
    """Mean per method as a bar, +/- 1 sd, with every individual run on top."""
    stats = aggregate(column)
    xs = np.arange(len(METHODS))

    ax.bar(xs, stats['mean'], width=0.55, color=[COLORS[m] for m in METHODS], alpha=0.85)
    ax.errorbar(xs, stats['mean'], yerr=stats['sd'].fillna(0), fmt='none',
                ecolor='#37474F', capsize=6, linewidth=1.4)

    for x, method in zip(xs, METHODS):
        rows = per_run[per_run['method'] == method]
        for batch, marker in BATCH_MARKERS.items():
            sub = rows[rows['batch'] == batch]
            if not len(sub):
                continue
            jitter = np.linspace(-0.16, 0.16, len(sub)) if len(sub) > 1 else np.array([0.0])
            ax.scatter(x + jitter, sub[column], marker=marker, s=44, color='white',
                       edgecolor='#263238', linewidth=1.2, zorder=5,
                       label=f'run in batch {batch}' if x == 0 else None)

    top = float((stats['mean'] + stats['sd'].fillna(0)).max())
    for x, method in zip(xs, METHODS):
        ax.text(x, stats.loc[method, 'mean'] + (stats.loc[method, 'sd'] or 0) + top * 0.04,
                fmt.format(stats.loc[method, 'mean']), ha='center', va='bottom',
                fontsize=11, fontweight='bold')

    if ref is not None:
        ax.axhline(ref, color='red', linestyle='--', linewidth=2, alpha=0.8, label=ref_label)

    ax.set_xticks(xs)
    ax.set_xticklabels([LABELS[m] for m in METHODS])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(top, ref or 0) * 1.28)
    ax.legend(loc='upper right', fontsize=9)
    return stats


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rt_stats = bar_with_runs(
    axes[0], 'rt_mean', 'Mean P90 per observation window (ms)',
    'Response Time Sisi Server (rata-rata antar run)',
    ref=SLO_THRESHOLD, ref_label=f'SLO {SLO_THRESHOLD:.0f} ms')

slo_stats = bar_with_runs(
    axes[1], 'slo_pct', 'Observation windows over SLO (%)',
    'Pelanggaran SLO per Jendela Observasi', fmt='{:.1f}%')

plt.tight_layout()
plt.show()

print(f'{"SERVER-SIDE RESPONSE TIME ACROSS RUNS":^96}')
print('=' * 96)
print(f'{"Method":<20} {"runs":>5} {"mean P90":>12} {"sd":>9} {"median P90":>12} '
      f'{"SLO viol %":>12} {"sd":>8}')
print('-' * 96)
med = aggregate('rt_med')
for method in METHODS:
    print(f'{LABELS[method]:<20} {int(rt_stats.loc[method, "n"]):>5} '
          f'{rt_stats.loc[method, "mean"]:>9.0f} ms {rt_stats.loc[method, "sd"]:>8.0f} '
          f'{med.loc[method, "mean"]:>9.0f} ms {slo_stats.loc[method, "mean"]:>11.1f}% '
          f'{slo_stats.loc[method, "sd"]:>7.1f}')
print('-' * 96)

## 5. Pelanggaran SLO per Tahap Beban

Rata-rata antar run, dipecah menurut kelima tahap `ramping-vus` k6. Inilah tempat
perbedaan antar metode paling terlihat: pada beban rendah semuanya patuh, dan yang
membedakan adalah seberapa cepat tiap metode pulih setelah puncak beban.

In [ ]:
stages = k6_results.LOAD_STAGES
stage_edges_min = np.cumsum([0] + [s['duration_min'] for s in stages])
stage_names = [f'{stage_edges_min[i]:.0f}-{stage_edges_min[i + 1]:.0f}m\n'
               f'({stages[i - 1]["target"] if i else 0}->{stages[i]["target"]} VUs)'
               for i in range(len(stages))]


def stage_violation_pct(df, lo_min, hi_min):
    """Share of windows in [lo, hi) minutes whose P90 exceeded the SLO."""
    window = df[(df['elapsed_min'] >= lo_min) & (df['elapsed_min'] < hi_min)]
    rt = window['response_time_ms']
    rt = rt[rt.notna() & (rt > 0)]
    if not len(rt):
        return np.nan
    return (rt > SLO_THRESHOLD).mean() * 100


stage_rows = []
for spec in specs:
    df = frames[(spec['method'], spec['run_id'])]
    for i in range(len(stages)):
        stage_rows.append({
            'method': spec['method'], 'run_id': spec['run_id'], 'batch': spec['batch'],
            'stage': i, 'viol_pct': stage_violation_pct(df, stage_edges_min[i],
                                                        stage_edges_min[i + 1]),
        })
per_stage = pd.DataFrame(stage_rows)
stage_mean = (per_stage.pivot_table(index='stage', columns='method', values='viol_pct',
                                    aggfunc='mean', observed=True)
              .reindex(columns=METHODS))
stage_sd = (per_stage.pivot_table(index='stage', columns='method', values='viol_pct',
                                  aggfunc='std', observed=True)
            .reindex(columns=METHODS))

fig, ax = plt.subplots(figsize=(15, 6))
xs = np.arange(len(stages))
width = 0.8 / len(METHODS)

for i, method in enumerate(METHODS):
    offset = (i - (len(METHODS) - 1) / 2) * width
    ax.bar(xs + offset, stage_mean[method], width, label=LABELS[method],
           color=COLORS[method], alpha=0.88,
           yerr=stage_sd[method].fillna(0), capsize=3,
           error_kw={'ecolor': '#37474F', 'linewidth': 1})
    for x, v in zip(xs + offset, stage_mean[method]):
        if v > 1:
            ax.text(x, v + 2, f'{v:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(xs)
ax.set_xticklabels(stage_names, fontsize=9)
ax.set_xlabel('Tahap beban k6 (per skenario, dua skenario berjalan bersamaan)')
ax.set_ylabel('Jendela melampaui SLO (%)')
ax.set_title('Pelanggaran SLO per Tahap Beban - rata-rata seluruh run (+/- 1 sd)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, 118)
ax.legend(fontsize=10, ncols=4, loc='upper left')

plt.tight_layout()
plt.show()

print('Mean SLO violation % per load stage (all runs)')
stage_table = stage_mean.copy()
stage_table.index = [n.replace('\n', ' ') for n in stage_names]
stage_table.columns = [LABELS[m] for m in METHODS]
stage_table.round(1)

## 6. Perilaku Scaling dan Biaya Replica

`rep_minutes` adalah integral jumlah replica terhadap waktu selama horizon bersama:
proksi biaya sumber daya. Jumlah scale-up dan scale-down menggambarkan seberapa
gelisah tiap metode - HPA hampir tidak pernah menurunkan replica dalam jendela ini,
sedangkan agent RL mengambil keputusan setiap iterasi.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))

rep_stats = bar_with_runs(axes[0], 'rep_minutes', 'Replica-minutes',
                          'Biaya Replica (integral replica x waktu)', fmt='{:,.0f}')
mean_rep = bar_with_runs(axes[1], 'rep_mean', 'Mean replica count',
                         'Rata-rata Jumlah Replica', fmt='{:.2f}')

# Scale-ups and scale-downs side by side: two different actions, so they get two
# bars rather than one summed "changes" number.
xs = np.arange(len(METHODS))
ups = aggregate('ups')
downs = aggregate('downs')
ax = axes[2]
ax.bar(xs - 0.2, ups['mean'], 0.38, label='Scale-ups', color='#EF6C00', alpha=0.88,
       yerr=ups['sd'].fillna(0), capsize=4, error_kw={'ecolor': '#37474F', 'linewidth': 1})
ax.bar(xs + 0.2, downs['mean'], 0.38, label='Scale-downs', color='#1E88E5', alpha=0.88,
       yerr=downs['sd'].fillna(0), capsize=4, error_kw={'ecolor': '#37474F', 'linewidth': 1})
for x, method in zip(xs, METHODS):
    ax.text(x - 0.2, ups.loc[method, 'mean'] + 0.6, f'{ups.loc[method, "mean"]:.1f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.text(x + 0.2, downs.loc[method, 'mean'] + 0.6, f'{downs.loc[method, "mean"]:.1f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(xs)
ax.set_xticklabels([LABELS[m] for m in METHODS], fontsize=9)
ax.set_ylabel('Actions per run')
ax.set_title('Aktivitas Scaling', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

hpa_cost = rep_stats.loc['hpa', 'mean']
print(f'{"REPLICA COST AND SCALING ACTIVITY (mean across runs)":^92}')
print('=' * 92)
print(f'{"Method":<20} {"rep-min":>10} {"sd":>8} {"vs HPA":>9} {"mean rep":>10} '
      f'{"ups":>7} {"downs":>7} {"max rep":>8}')
print('-' * 92)
rep_max = aggregate('rep_max')
for method in METHODS:
    print(f'{LABELS[method]:<20} {rep_stats.loc[method, "mean"]:>10.1f} '
          f'{rep_stats.loc[method, "sd"]:>8.1f} '
          f'{rep_stats.loc[method, "mean"] / hpa_cost:>8.0%} '
          f'{mean_rep.loc[method, "mean"]:>10.2f} {ups.loc[method, "mean"]:>7.1f} '
          f'{downs.loc[method, "mean"]:>7.1f} {rep_max.loc[method, "mean"]:>8.1f}')
print('-' * 92)
print('"vs HPA" is replica-minutes relative to the HPA baseline over the same horizon.')

## 7. Utilisasi CPU dan Memori

Utilisasi rata-rata pada horizon bersama. Nilai rendah bukan otomatis buruk: nilai
ini harus dibaca bersama biaya replica di atas, karena menaikkan replica pada beban
yang sama akan menurunkan utilisasi per pod.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))

cpu_stats = bar_with_runs(axes[0], 'cpu_mean', 'Mean CPU usage (%)',
                          'Utilisasi CPU', fmt='{:.1f}%')
mem_stats = bar_with_runs(axes[1], 'mem_mean', 'Mean memory usage (%)',
                          'Utilisasi Memori', fmt='{:.2f}%')

# CPU delivered per replica-minute: how much work each unit of cost carried.
per_run['cpu_per_replica'] = per_run['cpu_mean'] / per_run['rep_mean']
eff_stats = bar_with_runs(axes[2], 'cpu_per_replica', 'Mean CPU % per replica',
                          'CPU per Replica (efisiensi)', fmt='{:.2f}')

plt.tight_layout()
plt.show()

rps = aggregate('rps_mean')
print(f'{"UTILISATION (mean across runs)":^80}')
print('=' * 80)
print(f'{"Method":<20} {"CPU %":>9} {"sd":>7} {"Mem %":>9} {"sd":>7} {"RPS":>9} '
      f'{"CPU/replica":>12}')
print('-' * 80)
for method in METHODS:
    print(f'{LABELS[method]:<20} {cpu_stats.loc[method, "mean"]:>9.2f} '
          f'{cpu_stats.loc[method, "sd"]:>7.2f} {mem_stats.loc[method, "mean"]:>9.2f} '
          f'{mem_stats.loc[method, "sd"]:>7.2f} {rps.loc[method, "mean"]:>9.2f} '
          f'{eff_stats.loc[method, "mean"]:>12.2f}')
print('-' * 80)

## 8. Pola per Run

Bagian ini menampilkan tiap run apa adanya, bukan rata-ratanya, untuk memeriksa
apakah pola yang sama benar-benar berulang.

Menumpuk 24 fungsi tangga pada satu sumbu tidak terbaca, jadi pola ditampilkan
dengan dua cara: **heatmap** (satu baris per run, warna = jumlah replica) untuk
melihat keseluruhannya sekaligus, dan **satu panel per metode** yang menampilkan
keenam run metode tersebut beserta rata-ratanya.

In [ ]:
# One 15 s grid for every run, so trajectories from runs with slightly different
# sampling cadence can be averaged and compared point by point. 'ffill' keeps step
# semantics: a replica count holds until the next observation.
GRID_S = np.arange(0, HORIZON_S + 15, 15)
GRID_MIN = GRID_S / 60.0


def on_grid(df, column):
    s = pd.Series(df[column].values, index=df['elapsed_s'].values)
    s = s[~s.index.duplicated(keep='first')].sort_index()
    return s.reindex(GRID_S, method='ffill').bfill().values


# Runs in a fixed reading order: by method, then batch, then folder.
ordered_runs = []
for method in METHODS:
    for spec in specs:
        if spec['method'] == method:
            ordered_runs.append((method, spec['run_id'], spec['batch']))

stage_colors = ['#E3F2FD', '#BBDEFB', '#FFCDD2', '#C8E6C9', '#FFF9C4']


def shade_stages(ax):
    for i in range(len(stages)):
        ax.axvspan(stage_edges_min[i], stage_edges_min[i + 1], alpha=0.16,
                   color=stage_colors[i], zorder=0)


matrix = np.array([on_grid(frames[(m, r)], 'replica_state') for m, r, _ in ordered_runs])

fig, (ax_load, ax_heat) = plt.subplots(
    2, 1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [1, 5.5]})

vus = [0] + [s['target'] for s in stages]
ax_load.plot(stage_edges_min, vus, color='#2196F3', linewidth=2, marker='o')
ax_load.fill_between(stage_edges_min, vus, color='#2196F3', alpha=0.15)
ax_load.set_ylabel('VUs')
ax_load.set_ylim(0, max(vus) * 1.25)
ax_load.set_title('Jumlah Replica Sepanjang Waktu - Seluruh Run',
                  fontsize=14, fontweight='bold')

mesh = ax_heat.pcolormesh(np.append(GRID_MIN, GRID_MIN[-1] + 0.25),
                          np.arange(len(ordered_runs) + 1), matrix,
                          cmap='YlOrRd', vmin=1, vmax=10, shading='flat')
ax_heat.set_yticks(np.arange(len(ordered_runs)) + 0.5)
ax_heat.set_yticklabels([f'{r}   ({b})' for _, r, b in ordered_runs], fontsize=8)
ax_heat.invert_yaxis()

# Group separators and a method name against each block of rows.
start = 0
for method in METHODS:
    count = sum(1 for m, _, _ in ordered_runs if m == method)
    if start:
        ax_heat.axhline(start, color='white', linewidth=2.5)
    # The y axis is inverted, so row 0 sits at the top: an axes-fraction position
    # has to be flipped to match.
    ax_heat.text(-0.085, 1 - (start + count / 2) / len(ordered_runs),
                 LABELS[method], transform=ax_heat.transAxes, rotation=90,
                 ha='center', va='center', fontsize=10, fontweight='bold',
                 color=COLORS[method])
    start += count

ax_heat.set_xlabel('Elapsed Time (minutes)')
ax_heat.set_xticks(stage_edges_min)
ax_heat.set_xticklabels([f'{t:.0f}m' for t in stage_edges_min])
ax_heat.set_xlim(0, stage_edges_min[-1])

cbar = fig.colorbar(mesh, ax=[ax_load, ax_heat], pad=0.012)
cbar.set_label('Replica Count')
plt.show()

In [ ]:
# One panel per method: its runs as thin lines, its mean trajectory in bold. This
# is the "does the pattern repeat?" view - a method whose runs sit on top of each
# other behaves consistently, one whose runs fan out does not.
fig, axes = plt.subplots(len(METHODS), 1, figsize=(15, 12), sharex=True, sharey=True)

for ax, method in zip(axes, METHODS):
    shade_stages(ax)
    grids = []
    for m, run_id, batch in ordered_runs:
        if m != method:
            continue
        grid = on_grid(frames[(m, run_id)], 'replica_state')
        grids.append(grid)
        ax.step(GRID_MIN, grid, where='post', color=COLORS[method], alpha=0.35,
                linewidth=1.1)
    mean_grid = np.mean(grids, axis=0)
    ax.plot(GRID_MIN, mean_grid, color=COLORS[method], linewidth=2.8,
            label=f'mean of {len(grids)} runs')
    spread = np.mean(np.ptp(np.array(grids), axis=0))
    ax.set_title(f'{LABELS[method]}   -   mean {mean_grid.mean():.2f} replicas, '
                 f'spread between runs {spread:.1f} replicas on average',
                 fontsize=11, loc='left')
    ax.set_ylabel('Replicas')
    ax.set_ylim(0, 11)
    ax.set_yticks(range(0, 11, 2))
    ax.legend(loc='lower right', fontsize=9)

axes[-1].set_xlabel('Elapsed Time (minutes)')
axes[-1].set_xticks(stage_edges_min)
axes[-1].set_xticklabels([f'{t:.0f}m' for t in stage_edges_min])
axes[0].set_xlim(0, stage_edges_min[-1])
fig.suptitle('Jumlah Replica per Metode - seluruh run beserta rata-ratanya',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Response time the same way, on a log axis: the per-window P90 spans three orders
# of magnitude inside a single run, so a linear axis would flatten everything below
# the peak onto the baseline. The bold line is the median across runs at each point
# in time, not the mean - a single 30 s stall would drag a mean upward for the whole
# panel.
fig, axes = plt.subplots(len(METHODS), 1, figsize=(15, 12), sharex=True, sharey=True)

for ax, method in zip(axes, METHODS):
    shade_stages(ax)
    grids = []
    for m, run_id, batch in ordered_runs:
        if m != method:
            continue
        df = frames[(m, run_id)]
        mask = df['response_time_ms'].notna() & (df['response_time_ms'] > 0)
        ax.plot(df.loc[mask, 'elapsed_min'], df.loc[mask, 'response_time_ms'],
                linewidth=0.9, color=COLORS[method], alpha=0.32)
        grids.append(on_grid(df.loc[mask], 'response_time_ms'))

    ax.plot(GRID_MIN, np.median(np.array(grids), axis=0), color=COLORS[method],
            linewidth=2.6, label=f'median of {len(grids)} runs')
    ax.axhline(SLO_THRESHOLD, color='red', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'SLO {SLO_THRESHOLD:.0f} ms')
    ax.set_yscale('log')
    ax.set_title(f'{LABELS[method]}', fontsize=11, loc='left')
    ax.set_ylabel('P90 per window\n(ms, log)')
    ax.legend(loc='upper left', fontsize=9, ncols=2)

axes[-1].set_xlabel('Elapsed Time (minutes)')
axes[-1].set_xticks(stage_edges_min)
axes[-1].set_xticklabels([f'{t:.0f}m' for t in stage_edges_min])
axes[0].set_xlim(0, stage_edges_min[-1])
fig.suptitle('Response Time per Metode - seluruh run beserta mediannya',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Batch A vs Batch B

Kedua batch menjalankan profil beban yang sama terhadap kode yang sama, sehingga
selisih di bawah adalah ukuran seberapa besar kondisi klaster memengaruhi hasil.
Selama selisih itu sebanding dengan selisih antar metode, kesimpulan hanya boleh
ditarik di dalam satu batch.

Perhatikan baseline HPA: konfigurasinya identik di kedua batch, namun P90 k6-nya
turun dari ~2690 ms menjadi ~565 ms sementara P90 sisi servernya justru **naik**
dari ~181 ms menjadi ~811 ms. Kedua sisi bergerak berlawanan arah, dan itu hanya
mungkin bila sebagian besar latensi batch A terjadi **sebelum** request sampai ke
handler (antrean di ingress atau jaringan) sehingga tidak terlihat oleh histogram
aplikasi. Karena itu kolom k6 dan kolom Prometheus tidak boleh dianggap saling
mengonfirmasi: keduanya mengukur ruas yang berbeda dari jalur permintaan.

In [ ]:
k6_batch = k6.pivot_table(index='method', columns='batch', values='p90',
                          aggfunc='mean', observed=True).reindex(METHODS)
panels = [
    ('k6 P90 (ms)', k6_batch, '{:,.0f}'),
    ('Server-side mean P90 (ms)',
     per_run.pivot_table(index='method', columns='batch', values='rt_mean',
                         aggfunc='mean', observed=True).reindex(METHODS), '{:,.0f}'),
    ('SLO violation (%)',
     per_run.pivot_table(index='method', columns='batch', values='slo_pct',
                         aggfunc='mean', observed=True).reindex(METHODS), '{:.1f}'),
    ('Replica-minutes',
     per_run.pivot_table(index='method', columns='batch', values='rep_minutes',
                         aggfunc='mean', observed=True).reindex(METHODS), '{:,.0f}'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
xs = np.arange(len(METHODS))

for ax, (title, table, fmt) in zip(axes.flat, panels):
    for i, batch in enumerate(('A', 'B')):
        offset = (i - 0.5) * 0.4
        # Colour stays with the method; the batch is carried by the hatch, so the
        # two batches of one method never look like two different methods.
        ax.bar(xs + offset, table[batch], 0.38,
               color=[COLORS[m] for m in METHODS],
               alpha=0.9 if batch == 'B' else 0.45,
               hatch='' if batch == 'B' else '///', edgecolor='white')
        for x, method in zip(xs + offset, METHODS):
            ax.text(x, table.loc[method, batch], fmt.format(table.loc[method, batch]),
                    ha='center', va='bottom', fontsize=8)
    ax.set_xticks(xs)
    ax.set_xticklabels([LABELS[m] for m in METHODS], fontsize=9, rotation=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, table.max().max() * 1.22)

batch_handles = [
    plt.matplotlib.patches.Patch(facecolor='#9E9E9E', alpha=0.45, hatch='///',
                                 edgecolor='white',
                                 label=f'batch A ({k6_results.BATCH_DATES["A"]})'),
    plt.matplotlib.patches.Patch(facecolor='#9E9E9E', alpha=0.9, edgecolor='white',
                                 label=f'batch B ({k6_results.BATCH_DATES["B"]})'),
]
axes[0, 0].legend(handles=batch_handles, fontsize=9, loc='upper right')
fig.suptitle('Batch A vs Batch B', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'{"BATCH A vs BATCH B":^86}')
print('=' * 86)
print(f'{"Metric":<28} {"Method":<20} {"batch A":>11} {"batch B":>11} {"change":>10}')
print('-' * 86)
for title, table, fmt in panels:
    for method in METHODS:
        a, b = table.loc[method, 'A'], table.loc[method, 'B']
        change = (b - a) / a * 100 if a else float('nan')
        print(f'{title:<28} {LABELS[method]:<20} {fmt.format(a):>11} '
              f'{fmt.format(b):>11} {change:>9.0f}%')
    print('-' * 86)

## 10. Ringkasan

In [ ]:
final = pd.DataFrame({
    'Runs': aggregate('rt_mean')['n'].astype(int),
    'k6 P90 (ms)': overall['mean'].round(0),
    'k6 P90 sd': overall['sd'].round(0),
    'Server P90 mean (ms)': aggregate('rt_mean')['mean'].round(0),
    'SLO viol (%)': aggregate('slo_pct')['mean'].round(1),
    'SLO viol sd': aggregate('slo_pct')['sd'].round(1),
    'Replica-min': aggregate('rep_minutes')['mean'].round(1),
    'Mean replicas': aggregate('rep_mean')['mean'].round(2),
    'Scale-ups': aggregate('ups')['mean'].round(1),
    'Scale-downs': aggregate('downs')['mean'].round(1),
    'CPU (%)': aggregate('cpu_mean')['mean'].round(1),
})
final.index = [LABELS[m] for m in final.index]

print(f'{"SUMMARY - MEAN ACROSS ALL RUNS":^108}')
print('=' * 108)
for column in ('k6 P90 (ms)', 'Server P90 mean (ms)', 'SLO viol (%)', 'Replica-min'):
    ranking = final[column].sort_values()
    winner, best = ranking.index[0], ranking.iloc[0]
    runner, second = ranking.index[1], ranking.iloc[1]
    print(f'{column:<24} lowest: {winner:<20} {best:>9,.1f}   '
          f'next: {runner:<20} {second:>9,.1f}')
print('=' * 108)
print('Lower is better for every column above. Read them together: a method can buy')
print('latency with replicas, so the SLO columns only mean something next to the cost.')

final

### Cara membaca hasil ini

- **Rata-rata antar run menyembunyikan sebaran yang besar.** Simpangan baku P90 k6
  antar run berada pada orde ratusan milidetik, sehingga selisih beberapa puluh
  milidetik antar metode tidak dapat dibedakan dari derau. Yang layak dilaporkan
  adalah selisih yang konsisten arahnya pada seluruh run - lihat kembali panel per
  run di bagian 8.
- **Batch B jauh lebih cepat daripada batch A pada seluruh metode**, termasuk pada
  baseline HPA yang tidak berubah sama sekali. Ini menandakan pergeseran kondisi
  klaster, bukan perbaikan algoritme, sehingga membandingkan run batch A dengan run
  batch B akan menyesatkan.
- **P90 k6 dan P90 Prometheus tidak dapat dipertukarkan.** k6 mengukur di sisi klien
  dan menimbang per request; kolom Prometheus adalah rata-rata dari P90 per jendela
  observasi, dengan tiap jendela berbobot sama tanpa memandang jumlah request di
  dalamnya. Keduanya dilaporkan berdampingan, tidak pernah dijumlahkan.
- **Biaya replica adalah bagian dari hasil, bukan catatan kaki.** Metode dengan
  pelanggaran SLO paling rendah belum tentu unggul jika mencapainya dengan
  replica-minutes yang jauh lebih besar.